In [ ]:
# test to refine one dataset to see that we refine vibrations also, since they got stuck in an error


import importlib, refine_data as rd
importlib.reload(rd)

from pathlib import Path
run = Path("Data 2") / "2024-12-10 20-00-00 (1)"

meta = rd.refine_run_folder(run, overwrite=True)
print("meta vibration_error:", meta.get("vibration_error"))
print("refined files:", [p.name for p in run.glob("*refined*")])


=== Refining folder: Data 2\2024-12-10 20-00-00 (1) ===
meta vibration_error: None
refined files: ['CH1_ACCEL1Z1.refined.csv', 'CH2_ACCEL1Z2.refined.csv', 'GPS.latitude.refined.csv', 'GPS.longitude.refined.csv', 'GPS.speed.refined.csv', 'GPS.speed.refined.norm.csv', 'refine.refined.meta.json', 'vibration.refined.spikes.csv', 'vibration_segments.refined.features.csv']


In [ ]:
#
# Data check that files got refined. 

import pandas as pd
df = pd.read_csv(r"Data 2\2024-12-10 20-00-00 (1)\vibration_segments.refined.features.csv")
print(df.head())
print("segments:", len(df))


   seg   rms_ch1   peak_ch1    p2p_ch1   kurt_ch1    rms_ch2    peak_ch2  \
0    0  1.758377  12.115940  15.686207  16.828627  28.330193  217.628728   
1    1  0.955453   3.261916   6.465302   2.950373   0.992724    4.276363   
2    2  0.958111   3.234207   6.243188   2.732359   1.018399    3.284831   
3    3  1.012967   3.860393   7.431993   3.028233   1.021694    3.402500   
4    4  0.953846   3.501218   6.484565   3.018701   1.025947    3.874659   

      p2p_ch2   kurt_ch2     energy        peak  
0  220.695981  33.591183  30.088570  217.628728  
1    7.566522   3.097635   1.948177    4.276363  
2    6.476562   2.817953   1.976510    3.284831  
3    6.481002   2.918493   2.034661    3.860393  
4    7.055437   2.866166   1.979793    3.874659  
segments: 7200


In [ ]:
# debug code since 0 rail joints turned up in the data 


import numpy as np
import pandas as pd

# Läser RailJoint-punkter från Data 1
rj = pd.read_csv("Data 1/converted_coordinates_Resultat_RailJoint.csv")
rj.columns = rj.columns.str.strip()
rj_pts = rj[["Latitude","Longitude"]].dropna().to_numpy(float)

# Läser segmentens positioner (från din träningsfil)
segdf = pd.read_csv("segments_labeled.csv")
lat = segdf["Latitude"].to_numpy(float)
lon = segdf["Longitude"].to_numpy(float)

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    lat1 = np.radians(lat1); lon1 = np.radians(lon1)
    lat2 = np.radians(lat2); lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# min-avstånd per segment till närmaste RailJoint-punkt
mins = []
for la, lo in zip(lat, lon):
    mins.append(np.min(haversine_m(la, lo, rj_pts[:,0], rj_pts[:,1])))
mins = np.array(mins)

print("min dist to RailJoint (m):",
      "p50", np.percentile(mins,50),
      "p90", np.percentile(mins,90),
      "p99", np.percentile(mins,99),
      "min", mins.min())
print("Segments within 250m:", (mins <= 250).sum())
print("Segments within 400m:", (mins <= 400).sum())
print("Segments within 600m:", (mins <= 600).sum())

min dist to RailJoint (m): p50 58138.57640044594 p90 58160.97001398216 p99 58332.02028877875 min 35111.16002147523
Segments within 250m: 0
Segments within 400m: 0
Segments within 600m: 0


In [ ]:

# debug code to find railjoints 

import numpy as np, pandas as pd
from pathlib import Path
from pandas.errors import EmptyDataError

# RailJoint points
rj = pd.read_csv("Data 1/converted_coordinates_Resultat_RailJoint.csv")
rj.columns = rj.columns.str.strip()
rj_pts = rj[["Latitude","Longitude"]].dropna().to_numpy(float)

def haversine_m(lat1, lon1, lat2, lon2):
    R=6371000.0
    lat1=np.radians(lat1); lon1=np.radians(lon1)
    lat2=np.radians(lat2); lon2=np.radians(lon2)
    dlat=lat2-lat1; dlon=lon2-lon1
    a=np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2*R*np.arcsin(np.sqrt(a))



def read_single_col_safe(path):
    try:
        s = pd.read_csv(path, header=None).iloc[:, 0]
        return pd.to_numeric(s, errors="coerce").dropna().to_numpy(float)
    except EmptyDataError:
        return np.array([], dtype=float)
    except Exception:
        return np.array([], dtype=float)

def run_rj_stats(run_folder, radius=250):
    lat = read_single_col_safe(run_folder/"GPS.latitude.refined.csv")
    lon = read_single_col_safe(run_folder/"GPS.longitude.refined.csv")
    n = min(len(lat), len(lon))
    if n < 2:
        print("Skipping (empty GPS refined):", run_folder.name)
        return np.inf, np.inf, 0.0  # inga data -> inget coverage
    lat, lon = lat[:n], lon[:n]

    mins = np.array([haversine_m(la, lo, rj_pts[:,0], rj_pts[:,1]).min()
                     for la, lo in zip(lat, lon)])
    return mins.min(), np.percentile(mins, 50), (mins <= radius).mean()
data2 = Path("Data 2")
rows=[]
for run in sorted([p for p in data2.iterdir() if p.is_dir()]):
    if (run/"GPS.latitude.refined.csv").exists():
        dmin, d50, cov = run_rj_stats(run, radius=250)
        rows.append((run.name, dmin, d50, cov))

rows_sorted = sorted(rows, key=lambda x: x[1])
for r in rows_sorted[:10]:
    print(r)


Skipping (empty GPS refined): 2024-12-08 10-00-00 (1)
Skipping (empty GPS refined): 2024-12-14 10-00-00 (1)
Skipping (empty GPS refined): 2024-12-15 20-00-00 (1)
Skipping (empty GPS refined): 2024-12-16 02-01-58 (1)
Skipping (empty GPS refined): 2024-12-18 22-00-00 (1)
Skipping (empty GPS refined): 2024-12-22 06-00-00 (1)
Skipping (empty GPS refined): 2024-12-22 08-00-00 (1)
Skipping (empty GPS refined): 2024-12-22 14-37-21 (1)
Skipping (empty GPS refined): 2024-12-22 20-00-00 (1)
Skipping (empty GPS refined): 2024-12-23 00-00-00 (1)
Skipping (empty GPS refined): 2024-12-24 12-00-00 (2)
Skipping (empty GPS refined): 2024-12-25 04-00-00 (1)
Skipping (empty GPS refined): 2024-12-25 18-00-00 (1)
Skipping (empty GPS refined): 2024-12-25 22-00-00 (1)
Skipping (empty GPS refined): 2024-12-26 04-52-16 (1)
('2024-12-10 12-00-00 (1)', np.float64(0.22722114274122376), np.float64(11173.613550901766), np.float64(0.0615))
('2024-12-11 10-00-00 (1)', np.float64(2.235473405424952), np.float64(11180.4

In [ ]:

"""Automated track infrastructure recognition using vibration 

Select runs uses refine_data and preprocess the data befor selecting a run 



"""



import importlib, select_runs as sr
importlib.reload(sr)
manifest = sr.build_manifest()


=== Refining folder: Data 2\2024-12-08 10-00-00 (1) ===
[2024-12-08 10-00-00 (1)] score=0.000 refcov=0.000 dist_km=0.000 rj_cov=0.000 -> score<0.9;coverage<0.05;distance<0.4km
=== Refining folder: Data 2\2024-12-10 10-00-00 (1) ===
[2024-12-10 10-00-00 (1)] score=0.008 refcov=0.034 dist_km=15.879 rj_cov=0.000 -> score<0.9;coverage<0.05
=== Refining folder: Data 2\2024-12-10 12-00-00 (1) ===
[2024-12-10 12-00-00 (1)] score=0.319 refcov=0.353 dist_km=81.975 rj_cov=0.062 -> score<0.9
=== Refining folder: Data 2\2024-12-10 14-00-00 (1) ===
[2024-12-10 14-00-00 (1)] score=0.106 refcov=0.168 dist_km=44.675 rj_cov=0.000 -> score<0.9
=== Refining folder: Data 2\2024-12-10 16-00-00 (1) ===
[2024-12-10 16-00-00 (1)] score=0.202 refcov=0.412 dist_km=73.020 rj_cov=0.073 -> score<0.9
=== Refining folder: Data 2\2024-12-10 18-00-00 (1) ===
[2024-12-10 18-00-00 (1)] score=0.034 refcov=0.059 dist_km=8.679 rj_cov=0.000 -> score<0.9
=== Refining folder: Data 2\2024-12-10 20-00-00 (1) ===
[2024-12-10 20-

In [31]:
# since it takes forever to run select_runs I use the mainfest to rerun for adding more features to the data set



import json
import importlib
from pathlib import Path
import refine_data as rd

importlib.reload(rd)

manifest = json.loads(Path("selected_runs.json").read_text(encoding="utf-8"))
runs = [x["run"] for x in manifest.get("selected", [])]

print("Runs in manifest:", len(runs))

for r in runs:
    folder = Path("Data 2") / r
    if not folder.exists():
        print("SKIP (missing folder):", folder)
        continue

    print("\n--- Refining:", folder, "---")
    meta = rd.refine_run_folder(folder, overwrite=True)   # overwrite=True => skriv om features med nya kolumner
    if meta.get("vibration_error"):
        print("  VIBRATION ERROR:", meta["vibration_error"])
    else:
        print("  OK. Features:", meta.get("vibration_features"))

Runs in manifest: 6

--- Refining: Data 2\2024-12-10 20-00-00 (1) ---
=== Refining folder: Data 2\2024-12-10 20-00-00 (1) ===
  OK. Features: Data 2\2024-12-10 20-00-00 (1)\vibration_segments.refined.features.csv

--- Refining: Data 2\2024-12-10 22-00-00 (1) ---
=== Refining folder: Data 2\2024-12-10 22-00-00 (1) ===
  OK. Features: Data 2\2024-12-10 22-00-00 (1)\vibration_segments.refined.features.csv

--- Refining: Data 2\2024-12-11 04-00-00 (1) ---
=== Refining folder: Data 2\2024-12-11 04-00-00 (1) ===
  OK. Features: Data 2\2024-12-11 04-00-00 (1)\vibration_segments.refined.features.csv

--- Refining: Data 2\2024-12-11 06-00-00 (1) ---
=== Refining folder: Data 2\2024-12-11 06-00-00 (1) ===
  OK. Features: Data 2\2024-12-11 06-00-00 (1)\vibration_segments.refined.features.csv

--- Refining: Data 2\2024-12-11 00-00-00 (1) ---
=== Refining folder: Data 2\2024-12-11 00-00-00 (1) ===
  OK. Features: Data 2\2024-12-11 00-00-00 (1)\vibration_segments.refined.features.csv

--- Refining: 